# DeepWeeds Lab Day 2 ? Google Colab

Notebook n?p b?i: ch?y to?n b? tr?n Colab/GPU, l?u d? li?u v? k?t qu? tr?n Google Drive. C?c metric/s? li?u trong Excel v? b?o c?o ??u ???c ??c t? log ho?c prediction files th?t, kh?ng t? ?i?n ??c l??ng.

**Tr??c khi ch?y:** t?i `lab_code_bundle.zip` (???c t?o trong workspace) l?n `MyDrive/DeepWeedsLab/`, v? ??t `images.zip` t?i c?ng th? m?c. ?i?n MSSV v? h? t?n trong cell c?u h?nh. D?ng Runtime GPU.

Stages: EDA ? 5 backbone ? training ablation ? inference/TTA + latency ? ch?n theo val ? final v? baseline, 3 seed/test m?i seed m?t l?n ? export submission.


In [ ]:
!pip -q install timm openpyxl pandas numpy pillow matplotlib scikit-learn ptflops
import os, sys, json, hashlib, shutil, zipfile, time, re, unicodedata, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import timm

from google.colab import drive
drive.mount('/content/drive')

STUDENT_ID = '2A202602528'
STUDENT_NAME = 'VuongVietHoang'
DRIVE_ROOT = Path('/content/drive/MyDrive/DeepWeedsLab')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
assert STUDENT_ID and STUDENT_NAME, 'MSSV v? h? t?n kh?ng ???c ?? tr?ng.'
SLUG = re.sub(r'[^A-Za-z0-9_-]+', '_', f'{STUDENT_ID}_{STUDENT_NAME}').strip('_')
SUBMISSION_DIR = DRIVE_ROOT / 'submissions' / SLUG

print('Python:', platform.python_version(), '| torch:', torch.__version__, '| timm:', timm.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('H?y ch?n Runtime > Change runtime type > T4 GPU (ho?c GPU kh? d?ng).')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Gi?i n?n source code ?? chu?n b? trong workspace. Kh?ng clone m? starter pseudo-code.
BUNDLE_ZIP = DRIVE_ROOT / 'lab_code_bundle.zip'
if not BUNDLE_ZIP.exists():
    raise FileNotFoundError(f'T?i lab_code_bundle.zip t? workspace l?n Google Drive t?i: {BUNDLE_ZIP}')
CODE_ROOT = Path('/content/lab_code_bundle')
if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)
CODE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    zf.extractall(CODE_ROOT)

# Accept both supported archives: modules under starter/, or flattened beside eval.py.
eval_candidates = list(CODE_ROOT.rglob('eval.py'))
if not eval_candidates:
    raise RuntimeError(f'Bundle kh?ng c? eval.py. Archive hi?n c?: {[p.name for p in CODE_ROOT.iterdir()]}')
CODE_ROOT = eval_candidates[0].parent
TRAIN_DIR = CODE_ROOT / 'starter'
if not (TRAIN_DIR / 'train.py').exists():
    if (CODE_ROOT / 'train.py').exists():
        TRAIN_DIR = CODE_ROOT
    else:
        train_candidates = list(CODE_ROOT.rglob('train.py'))
        if train_candidates:
            TRAIN_DIR = train_candidates[0].parent
        else:
            raise RuntimeError(f'Bundle kh?ng c? train.py. Archive hi?n c?: {[str(p.relative_to(CODE_ROOT)) for p in CODE_ROOT.rglob("*") if p.is_file()][:30]}')
if not (CODE_ROOT / 'eval.py').exists():
    raise RuntimeError(f'eval.py kh?ng n?m c?nh bundle root ?? nh?n di?n: {CODE_ROOT}')
sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(TRAIN_DIR))
from dataset import load_split, check_split, build_transforms, make_loader
from train import Config, run, inference_logits
import eval as ev
from model import build_model
from inference import fit_temperature

# C?c file t?n ch? ???c gi? tr?n Drive; ?nh gi?i n?n v? cache loader n?m tr?n ??a Colab.
RUNS_DIR = DRIVE_ROOT / 'runs' / SLUG
PRED_DIR = DRIVE_ROOT / 'predictions' / SLUG
CURVES_DIR = DRIVE_ROOT / 'curves' / SLUG
for p in (RUNS_DIR, PRED_DIR, CURVES_DIR):
    p.mkdir(parents=True, exist_ok=True)
print('Submission target:', SUBMISSION_DIR)


In [ ]:
# Data: images.zip tr?n Drive, CSV fold 0 l?y t? repo g?c DeepWeeds.
IMG_ZIP = DRIVE_ROOT / 'images.zip'
if not IMG_ZIP.exists():
    raise FileNotFoundError(f'??t images.zip v?o {IMG_ZIP}')
md5 = hashlib.md5()
with IMG_ZIP.open('rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        md5.update(chunk)
actual_md5 = md5.hexdigest()
print('images.zip MD5:', actual_md5)
if actual_md5 != 'b7b30f96d466fba86016aa5a26606e0f':
    raise ValueError('MD5 images.zip kh?ng kh?p v?i README c?a lab.')

DATA_ROOT = Path('/content/deepweeds_data')
IMAGES_DIR = DATA_ROOT / 'images'
LABELS_DIR = DATA_ROOT / 'labels'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)
if not any(IMAGES_DIR.iterdir()):
    import zipfile
    with zipfile.ZipFile(IMG_ZIP) as zf:
        zf.extractall(IMAGES_DIR)

base = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for filename in ('labels.csv', 'train_subset0.csv', 'val_subset0.csv', 'test_subset0.csv'):
    target = LABELS_DIR / filename
    if not target.exists():
        from urllib.request import urlretrieve
        urlretrieve(f'{base}/{filename}', target)

# N?u archive ch?a th? m?c con, x?c nh?n t?t c? filename n?m trong ??ng image root v? ?nh x? t?n.
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
if not all((IMAGES_DIR / str(name)).is_file() for name in pd.concat([train_df, val_df, test_df])['Filename'].unique()):
    found = {p.name: p for p in IMAGES_DIR.rglob('*') if p.is_file()}
    missing = [n for n in pd.concat([train_df, val_df, test_df])['Filename'].unique() if str(n) not in found]
    if missing:
        raise FileNotFoundError(f'Archive thi?u ?nh theo CSV, v? d?: {missing[:5]}')
    # DeepWeeds filenames duy nh?t; link/copy t?ng ?nh v? image root ?? d?ng ??ng giao di?n Dataset.
    for filename, source in found.items():
        target = IMAGES_DIR / filename
        if source != target and not target.exists():
            shutil.copy2(source, target)

split_stats = check_split(train_df, val_df, test_df, IMAGES_DIR)
print(json.dumps(split_stats, indent=2))


## EDA b?t bu?c
Ki?m tra ???c th?c hi?n tr?n fold 0 tr??c hu?n luy?n. Cell t?o ph?n b? l?p v? montage ?nh minh ho? tr?n train; l?u v?o th? m?c submission khi export.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

CLASS_NAMES = ev.CLASS_NAMES
eda_rows = []
for split, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    counts = df['Label'].value_counts().reindex(range(9), fill_value=0)
    for label, count in counts.items():
        eda_rows.append({'split': split, 'label': int(label), 'class': CLASS_NAMES[int(label)], 'n': int(count)})
eda_df = pd.DataFrame(eda_rows)
fig, ax = plt.subplots(figsize=(13, 5))
for split, part in eda_df.groupby('split'):
    ax.bar(np.arange(9) + {'train': -0.25, 'val': 0, 'test': 0.25}[split], part['n'], width=0.25, label=split)
ax.set_xticks(range(9), CLASS_NAMES, rotation=45, ha='right')
ax.set_ylabel('?nh')
ax.set_title('DeepWeeds fold 0 ? s? ?nh theo l?p')
ax.legend()
fig.tight_layout()
EDA_DIR = Path('/content/eda')
EDA_DIR.mkdir(exist_ok=True)
eda_plot = EDA_DIR / 'class_distribution.png'
fig.savefig(eda_plot, dpi=180)
plt.show()

# Ch?n t?i ?a 3 v? d? m?i l?p; m?i ?nh ??c tr?c ti?p t? fold train.
fig, axes = plt.subplots(9, 3, figsize=(9, 21))
for label, axrow in enumerate(axes):
    samples = train_df[train_df.Label == label].head(3)
    for j, ax in enumerate(axrow):
        ax.axis('off')
        if j < len(samples):
            filename = str(samples.iloc[j].Filename)
            ax.imshow(Image.open(IMAGES_DIR / filename).convert('RGB'))
            ax.set_title(filename, fontsize=7)
    axrow[0].set_ylabel(CLASS_NAMES[label])
fig.tight_layout()
eda_montage = EDA_DIR / 'sample_images.png'
fig.savefig(eda_montage, dpi=160)
plt.show()
print('Train imbalance max/min:', train_df.Label.value_counts().max() / train_df.Label.value_counts().min())


## 1. Backbone comparison ? 5 ki?n tr?c
C?ng recipe, seed, fold v? budget epoch. Colab c? th? ng?t phi?n; m?i l?n ch?y s? b? qua exp_id ?? c? summary.json, gi?p ti?p t?c ? phi?n sau. B?t GPU v? gi? RUNS_DIR tr?n Drive.


In [ ]:
BASE = dict(
    fold=0, init='finetune', img_size=224, aug='basic', sampler=None, mix=None,
    loss='ce', epochs=12, batch_size=32, lr_backbone=1e-4, lr_head=1e-3,
    weight_decay=0.05, amp=True, num_workers=2,
    images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR),
    out_dir=str(RUNS_DIR), pred_dir=str(PRED_DIR), save_test_predictions=False,
)

def run_exp(params):
    cfg = Config(**params)
    root = Path(cfg.out_dir) / cfg.exp_id / f'seed{cfg.seed}'
    summary_file = root / 'summary.json'
    config_file = root / 'config.json'
    old_cfg = json.loads(config_file.read_text()) if config_file.exists() else None
    requested_cfg = __import__('dataclasses').asdict(cfg)
    test_file = Path(cfg.pred_dir) / f'{cfg.exp_id}_seed{cfg.seed}_test.csv'
    if cfg.save_test_predictions and test_file.exists() and old_cfg != requested_cfg:
        raise RuntimeError(f'Test prediction already exists for {cfg.exp_id}/seed{cfg.seed}; do not rerun test with a different config: {test_file}')
    if cfg.save_test_predictions and test_file.exists() and not summary_file.exists():
        raise RuntimeError(f'Test predictions exist but the run summary is missing. Preserve this one-time test result and inspect {root}; do not rerun it.')
    if summary_file.exists() and old_cfg == requested_cfg:
        saved = json.loads(summary_file.read_text())
        return {**saved, 'best_macro_f1': saved['val']['macro_f1'], 'best_top1': saved['val']['top1'],
                'best_ece': saved['val']['ece'], 'params_M': saved['params_M'],
                'gmac': saved.get('gmac', np.nan), 'checkpoint': str(root / 'best.pth'), 'out_dir': str(root)}
    return run(cfg)

BACKBONES = [
    ('B01', 'resnet50', 'ResNet family'),
    ('B02', 'resnext50_32x4d', 'ResNeXt family'),
    ('B03', 'convnext_tiny', 'ConvNeXt family'),
    ('B04', 'deit_small_patch16_224', 'Transformer'),
    ('B05', 'efficientnet_b0', 'Lightweight'),
]
backbone_results = []
for exp_id, backbone, family in BACKBONES:
    result = run_exp({**BASE, 'exp_id': exp_id, 'backbone': backbone, 'seed': 0})
    backbone_results.append({**result, 'backbone': backbone, 'family': family})
    print(exp_id, result['best_macro_f1'], result['out_dir'])

ABLATION_BACKBONE = max(backbone_results, key=lambda r: r['best_macro_f1'])['backbone']
print('Backbone selected by val for ablations:', ABLATION_BACKBONE)


## 2. Training ablation ? initialization, augmentation, loss, sampling, combination
M?i T01?T06 ch? thay m?t y?u t? so v?i T00. T07 l? th? nghi?m k?t h?p ?? khai b?o ri?ng. S?ng l?c d?ng m?t seed; final d?ng ?t nh?t ba seed.


In [ ]:
TRAINING_CONFIGS = [
    ('T00', dict()),
    ('T01', dict(init='scratch')),
    ('T02', dict(init='frozen')),
    ('T03', dict(aug='color')),
    ('T04', dict(loss='focal', focal_gamma=2.0)),
    ('T05', dict(loss='ls', label_smoothing=0.1)),
    ('T06', dict(sampler='balanced')),
    ('T07', dict(aug='color', loss='focal', focal_gamma=2.0, mix='cutmix')),
]
TRAIN_AXIS = {
    'T00': 'baseline', 'T01': 'A: initialization', 'T02': 'A: initialization',
    'T03': 'B: augmentation', 'T04': 'C: loss', 'T05': 'C: loss',
    'T06': 'D: balanced sampler', 'T07': 'combination: augmentation + loss + CutMix',
}
training_results = []
for exp_id, changes in TRAINING_CONFIGS:
    result = run_exp({**BASE, 'exp_id': exp_id, 'backbone': ABLATION_BACKBONE, 'seed': 0, **changes})
    training_results.append({**result, 'changes': changes, 'axis': TRAIN_AXIS[exp_id]})
    print(exp_id, result['best_macro_f1'], changes)

# Candidate/final selection is based only on validation macro-F1. No test file is read here.
selected_train = max(training_results, key=lambda r: r['best_macro_f1'])
selected_train_id = selected_train['exp_id']
selected_changes = dict(selected_train['changes'])
print('Selected training recipe by val macro-F1:', selected_train_id, selected_changes)


## 3. Inference methods ? val selection and measured latency
So s?nh one-view, horizontal-flip TTA, five-crop TTA v? temperature scaling. NLL/ECE/T ch? ???c fit tr?n validation. Latency ?o 10 warmup + 50 l?n, c? ??ng b? GPU, batch=1, kh?ng t?nh preprocessing.


In [ ]:
def load_saved_model(exp_id, seed=0, backbone='resnet50'):
    ckpt_path = RUNS_DIR / exp_id / f'seed{seed}' / 'best.pth'
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    net = build_model(backbone, pretrained=False, num_classes=9, init='finetune')
    net.load_state_dict(ckpt['model'])
    return net.to('cuda').eval()

selected_model = load_saved_model(selected_train_id, backbone=ABLATION_BACKBONE)
val_loader = make_loader(val_df, IMAGES_DIR, build_transforms(False, 224, 'basic'), 32, False, num_workers=2)

def collect_logits(model, loader, method):
    ys, all_logits = [], []
    model.eval()
    with torch.inference_mode():
        for x, y, _ in loader:
            x = x.cuda(non_blocking=True)
            logits = inference_logits(model, x, method)
            all_logits.append(logits.float().cpu().numpy())
            ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(all_logits)

def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(axis=1, keepdims=True)

def metric_row(y, logits):
    p = softmax_np(logits)
    m = ev.compute_metrics(y, p.argmax(1), p)
    return p, m

def latency_ms(model, method, size=224, batch_size=1, warmup=10, iters=50, temperature=None):
    x = torch.randn(batch_size, 3, size, size, device='cuda')
    def fn():
        with torch.inference_mode():
            logits = inference_logits(model, x, method)
            if temperature is not None:
                torch.softmax(logits / temperature, dim=1)
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize(); times.append((time.perf_counter() - t0) * 1000)
    return {'p50_ms': float(np.percentile(times, 50)), 'p95_ms': float(np.percentile(times, 95)),
            'p99_ms': float(np.percentile(times, 99)), 'images_per_s': float(batch_size * 1000 / np.percentile(times, 50)),
            'batch': batch_size, 'gpu': torch.cuda.get_device_name(0), 'dtype': 'FP32', 'batch': 1, 'img_size': size,
            'warmup': warmup, 'iterations': iters, 'preprocessing_included': False}

inference_results = []
val_cache = {}
for method in ('single', 'hflip', 'fivecrop'):
    yv, logits = collect_logits(selected_model, val_loader, method)
    p, m = metric_row(yv, logits)
    lat = latency_ms(selected_model, method)
    inference_results.append({'exp_id': 'I' + str(len(inference_results)+1).zfill(2), 'method': method,
        'model_checkpoint': selected_train_id, 'k_views': {'single':1,'hflip':2,'fivecrop':5}[method],
        'macro_f1_val': m['macro_f1'], 'top1_val': m['top1'], 'ece_val': m['ece'],
        **lat, 'relative_cost_vs_I00': lat['p50_ms'] / max(inference_results[0]['p50_ms'], 1e-9) if inference_results else 1.0})
    val_cache[method] = (yv, logits, p)

# Fit temperature only on val one-view logits, then compare ECE/metrics after calibration.
yv, single_logits, single_probs = val_cache['single']
temperature = fit_temperature(single_logits, yv)
temp_probs = softmax_np(single_logits / temperature)
temp_metrics = ev.compute_metrics(yv, temp_probs.argmax(1), temp_probs)
temp_lat = latency_ms(selected_model, 'single', temperature=temperature)
inference_results.append({'exp_id':'I04', 'method':f'temperature_scaling(T={temperature:.4f})',
    'model_checkpoint':selected_train_id, 'k_views':1, 'macro_f1_val':temp_metrics['macro_f1'],
    'top1_val':temp_metrics['top1'], 'ece_val':temp_metrics['ece'], **temp_lat,
    'relative_cost_vs_I00':temp_lat['p50_ms']/max(inference_results[0]['p50_ms'],1e-9)})

# Select by validation macro-F1; tie-break on ECE, then p95. Keep the selected method for all final seeds.
selected_inference = max(inference_results, key=lambda r: (r['macro_f1_val'], -r['ece_val'], -r['p95_ms']))
if selected_inference['method'].startswith('temperature_scaling'):
    FINAL_INFERENCE_METHOD, FINAL_TEMPERATURE = 'single', True
else:
    FINAL_INFERENCE_METHOD, FINAL_TEMPERATURE = selected_inference['method'], False
print('Selected inference on val:', selected_inference['method'], '=>', FINAL_INFERENCE_METHOD, 'temperature:', FINAL_TEMPERATURE)


## 4. Final + baseline ? test m?t l?n m?i seed
Ch?t c?u h?nh t? validation. Ch?y final recipe v? baseline T00 cho seed 0,1,2. `run()` ch? ??c test trong l??t cu?i n?y, l?u calibrated test predictions v? (n?u d?ng temperature scaling) uncalibrated predictions t? c?ng m?t l?n forward.


In [ ]:
FINAL_EXP_ID = 'F01'
FINAL_CONFIG = {**BASE, 'exp_id': FINAL_EXP_ID, 'backbone': ABLATION_BACKBONE, 'seed': 0,
                **selected_changes, 'inference_method': FINAL_INFERENCE_METHOD,
                'temperature_scaling': FINAL_TEMPERATURE}

final_results, baseline_results = [], []
for seed in (0, 1, 2):
    # Baseline T00, one-view, no temperature scaling.
    base_result = run_exp({**BASE, 'exp_id':'T00', 'backbone':ABLATION_BACKBONE, 'seed':seed,
                           'save_test_predictions':True, 'inference_method':'single', 'temperature_scaling':False})
    baseline_results.append(base_result)
    # Final recipe/method selected only from val.
    final_params = {**FINAL_CONFIG, 'seed':seed, 'save_test_predictions':True}
    final_result = run_exp(final_params)
    final_results.append(final_result)
    print('seed', seed, 'baseline', base_result['best_macro_f1'], 'final', final_result['best_macro_f1'])


## 5. Xu?t results.xlsx, report.md v? th? m?c submission
Workbook g?m ??ng c?c sheet/c?t GUIDE.md ?6.1 y?u c?u, c?ng sheet EDA/Confusion h? tr? b?o c?o. Sheet s? li?u ch? d?a tr?n k?t qu? th?t ?? l?u; thi?u experiment th? notebook b?o l?i thay v? ch?n s? m?c ??nh.


In [ ]:
# Validate prediction files against the untouched fold-0 reference CSVs.
final_pred_paths = [PRED_DIR / f'F01_seed{s}_test.csv' for s in (0,1,2)]
base_pred_paths = [PRED_DIR / f'T00_seed{s}_test.csv' for s in (0,1,2)]
for p in final_pred_paths + base_pred_paths:
    if not p.exists(): raise FileNotFoundError(f'Missing final prediction: {p}')
    pred = ev.read_pred(str(p))
    ev.check_against_csv(pred, str(LABELS_DIR / 'test_subset0.csv'), 'test')

def pred_metrics(paths):
    preds = [ev.read_pred(str(p)) for p in paths]
    ms = [ev.compute_metrics(p.y_true, p.y_pred, p.probs) for p in preds]
    return preds, ms

final_preds, final_ms = pred_metrics(final_pred_paths)
base_preds, base_ms = pred_metrics(base_pred_paths)

# Build Backbones rows with exact validation metrics and measured batch-1 latency.
backbone_rows = []
backbone_latencies = {}
for item in backbone_results:
    model = load_saved_model(item['exp_id'], 0, item['backbone'])
    lat = latency_ms(model, 'single')
    backbone_latencies[item['exp_id']] = lat
    s = json.loads((Path(item['out_dir']) / 'summary.json').read_text())
    backbone_rows.append({'exp_id':item['exp_id'], 'backbone':item['backbone'], 'pretrained_tag':item['pretrained_tag'],
        'params_M':item['params_M'], 'GMAC':item.get('gmac', np.nan), 'resolution_px':224,
        'epoch':item['best_epoch'], 'seed':0, 'macro_f1_val':s['val']['macro_f1'], 'top1_val':s['val']['top1'],
        'train_seconds_per_epoch':s['epoch_seconds_mean'], 'latency_batch1_ms':lat['p50_ms'], 'notes':item['family']})

# Read all T records; ensure ? baseline and rare-class F1 are sourced from prediction CSVs.
training_rows = []
t00_f1 = next(x['best_macro_f1'] for x in training_results if x['exp_id']=='T00')
for item in training_results:
    cfg = json.loads((Path(item['out_dir']) / 'config.json').read_text())
    val_pred_path = PRED_DIR / f"{item['exp_id']}_seed0_val.csv"
    pred = ev.read_pred(str(val_pred_path))
    met = ev.compute_metrics(pred.y_true, pred.y_pred, pred.probs)
    training_rows.append({'exp_id':item['exp_id'], 'backbone':cfg['backbone'], 'changed_axis':item['axis'],
        'difference_from_T00':json.dumps(item['changes'], ensure_ascii=False) if item['changes'] else 'baseline',
        'seed':0, 'macro_f1_val':met['macro_f1'], 'top1_val':met['top1'], 'delta_macro_f1_vs_T00':met['macro_f1']-t00_f1,
        'rare_class_f1_min':float(np.min(met['f1'][:8])), 'notes':'single screening seed; compare cautiously'})

# Final rows: each seed and a mean/std summary row. std uses ddof=1.
final_rows=[]
for label, paths, metrics in [('F01',final_pred_paths,final_ms),('T00',base_pred_paths,base_ms)]:
    run_results = final_results if label=='F01' else baseline_results
    for seed, (path, m, run_result) in enumerate(zip(paths, metrics, run_results)):
        final_rows.append({'exp_id':label,'configuration':json.dumps(FINAL_CONFIG if label=='F01' else {'backbone':ABLATION_BACKBONE,'recipe':'baseline','inference':'single'}, ensure_ascii=False),
            'seed':seed,'macro_f1_val':run_result['best_macro_f1'],'macro_f1_test':m['macro_f1'],'top1_test':m['top1'],'ece_test':m['ece'],'row_type':'seed'})
    for key in ('macro_f1_val','macro_f1_test','top1_test','ece_test'):
        vals = [r[key] for r in final_rows if r['exp_id']==label and r['row_type']=='seed']
        if key == 'macro_f1_val': val_mean, val_std = float(np.mean([x['best_macro_f1'] for x in run_results])), float(np.std([x['best_macro_f1'] for x in run_results], ddof=1))
        else: val_mean, val_std = float(np.mean(vals)), float(np.std(vals, ddof=1))
        if key == 'macro_f1_val':
            final_rows.append({'exp_id':label,'configuration':'mean ? sample std over seeds','seed':'mean ? std','macro_f1_val':f'{val_mean:.4f} ? {val_std:.4f}','macro_f1_test':None,'top1_test':None,'ece_test':None,'row_type':'summary'})
        elif key == 'macro_f1_test':
            final_rows[-1]['macro_f1_test'] = f'{val_mean:.4f} ? {val_std:.4f}'
        elif key == 'top1_test': final_rows[-1]['top1_test'] = f'{val_mean:.4f} ? {val_std:.4f}'
        elif key == 'ece_test': final_rows[-1]['ece_test'] = f'{val_mean:.4f} ? {val_std:.4f}'

# Per-class mean across seeds for final and baseline.
per_class_rows=[]
for label, metrics in [('F01',final_ms),('T00',base_ms)]:
    for i, name in enumerate(CLASS_NAMES):
        per_class_rows.append({'exp_id':label,'class':name,
            'test_support_mean':float(np.mean([m['support'][i] for m in metrics])),
            'precision_mean':float(np.mean([m['precision'][i] for m in metrics])),
            'recall_mean':float(np.mean([m['recall'][i] for m in metrics])),
            'f1_mean':float(np.mean([m['f1'][i] for m in metrics]))})

inference_rows=[]
for r in inference_results:
    inference_rows.append({'exp_id':r['exp_id'],'method':r['method'],'model_checkpoint':r['model_checkpoint'],'K_views_or_models':r['k_views'],
        'macro_f1_val':r['macro_f1_val'],'top1_val':r['top1_val'],'ECE_val':r['ece_val'],
        'latency_p50_ms':r['p50_ms'],'latency_p95_ms':r['p95_ms'],'latency_p99_ms':r['p99_ms'],
        'throughput_images_s':r['images_per_s'],'relative_cost_vs_I00':r['relative_cost_vs_I00']})

latency_rows=[]
for r in inference_results:
    latency_rows.append({'configuration':r['exp_id']+' '+r['method'],'GPU':r['gpu'],'dtype':r['dtype'],'batch':r['batch'],
        'img_size_px':r['img_size'],'BN_fused':'no','p50_ms':r['p50_ms'],'p95_ms':r['p95_ms'],
        'p99_ms':r['p99_ms'],'images_per_s':r['images_per_s']})
for row in backbone_rows:
    lat=backbone_latencies[row['exp_id']]
    latency_rows.append({'configuration':row['exp_id']+' single','GPU':lat['gpu'],'dtype':lat['dtype'],'batch':1,
        'img_size_px':224,'BN_fused':'no','p50_ms':lat['p50_ms'],'p95_ms':lat['p95_ms'],'p99_ms':lat['p99_ms'],
        'images_per_s':lat['images_per_s']})
batch32_latency = latency_ms(selected_model, FINAL_INFERENCE_METHOD, batch_size=32)
latency_rows.append({'configuration':'F01 selected method','GPU':batch32_latency['gpu'],'dtype':'FP32','batch':32,
    'img_size_px':224,'BN_fused':'no','p50_ms':batch32_latency['p50_ms'],'p95_ms':batch32_latency['p95_ms'],
    'p99_ms':batch32_latency['p99_ms'],'images_per_s':batch32_latency['images_per_s']})

summary_rows=[]
for r in backbone_rows:
    summary_rows.append({'category':'backbone','exp_id':r['exp_id'],'configuration':r['backbone'],'macro_f1_val':r['macro_f1_val'],'top1_val':r['top1_val'],'latency_p95_ms':backbone_latencies[r['exp_id']]['p95_ms'],'params_M':r['params_M']})
for r in training_rows:
    summary_rows.append({'category':'training','exp_id':r['exp_id'],'configuration':r['difference_from_T00'],'macro_f1_val':r['macro_f1_val'],'top1_val':r['top1_val'],'latency_p95_ms':np.nan,'params_M':np.nan})
for r in inference_rows:
    summary_rows.append({'category':'inference','exp_id':r['exp_id'],'configuration':r['method'],'macro_f1_val':r['macro_f1_val'],'top1_val':r['top1_val'],'latency_p95_ms':r['latency_p95_ms'],'params_M':np.nan})
summary_df=pd.DataFrame(summary_rows).sort_values('macro_f1_val',ascending=False).head(10).reset_index(drop=True)

# Confusion matrix from average final-seed probabilities. It is descriptive only and is not used for selection.
mean_probs = np.mean([p.probs for p in final_preds], axis=0)
y_test = final_preds[0].y_true
cm = ev.confusion_matrix(y_test, mean_probs.argmax(1))
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(9), CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(9), CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('F01 confusion matrix ? mean probabilities')
fig.tight_layout()
confusion_plot = EDA_DIR / 'confusion_F01.png'
fig.savefig(confusion_plot, dpi=180)
plt.close(fig)

# Exact GUIDE.md ?6.1 sheet names and required columns.
frames = {
 'Backbones':pd.DataFrame(backbone_rows), 'Training':pd.DataFrame(training_rows),
 'Inference':pd.DataFrame(inference_rows), 'Final':pd.DataFrame(final_rows),
 'PerClass':pd.DataFrame(per_class_rows), 'Latency':pd.DataFrame(latency_rows),
 'Summary':summary_df, 'EDA':eda_df, 'Confusion':cm_df.reset_index(names='true_class')
}

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
xlsx_path=SUBMISSION_DIR/'results.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    for sheet, frame in frames.items():
        frame.to_excel(writer, sheet_name=sheet, index=False)
    wb=writer.book
    for ws in wb.worksheets:
        ws.freeze_panes='A2'
        ws.auto_filter.ref=ws.dimensions
        for col in ws.columns:
            letter=col[0].column_letter
            width=min(48,max(12,max((len(str(cell.value)) if cell.value is not None else 0) for cell in col)+2))
            ws.column_dimensions[letter].width=width
        for cell in ws[1]:
            cell.font=__import__('openpyxl').styles.Font(bold=True,color='FFFFFF')
            cell.fill=__import__('openpyxl').styles.PatternFill('solid',fgColor='1F4E78')
        metric_col=next((c.column for c in ws[1] if c.value == 'macro_f1_val'),None)
        if metric_col and ws.max_row > 1:
            values=[ws.cell(r,metric_col).value for r in range(2,ws.max_row+1)]
            numeric=[(r,v) for r,v in enumerate(values,start=2) if isinstance(v,(int,float)) and np.isfinite(v)]
            if numeric:
                best_row=max(numeric,key=lambda item:item[1])[0]
                for cell in ws[best_row]:
                    cell.fill=__import__('openpyxl').styles.PatternFill('solid',fgColor='E2F0D9')

# Copy traceable curves and prediction files, but never dataset or checkpoints.
curves_out=SUBMISSION_DIR/'curves'; curves_out.mkdir(exist_ok=True)
for exp_id in [x[0] for x in BACKBONES] + [x[0] for x in TRAINING_CONFIGS] + ['F01']:
    source=RUNS_DIR/exp_id/'seed0'/'curve.png'
    if source.exists(): shutil.copy2(source, curves_out/f'{exp_id}_training.png')
(SUBMISSION_DIR/'predictions').mkdir(exist_ok=True)
for source in PRED_DIR.glob('*.csv'):
    if any(source.name.startswith(prefix) for prefix in ('F01_seed','F01_uncal_seed','T00_seed','T00_uncal_seed')):
        shutil.copy2(source,SUBMISSION_DIR/'predictions'/source.name)


In [ ]:
# Package runnable source, notebook, EDA evidence and truthful report.
code_out=SUBMISSION_DIR/'code'; code_out.mkdir(exist_ok=True)
shutil.copy2(CODE_ROOT/'eval.py',code_out/'eval.py')
for source in TRAIN_DIR.glob('*.py'):
    shutil.copy2(source,code_out/source.name)
# Include the Colab notebook supplied in the uploaded source bundle.
if (CODE_ROOT/'lab_day2_colab.ipynb').exists():
    shutil.copy2(CODE_ROOT/'lab_day2_colab.ipynb',code_out/'lab_day2_colab.ipynb')
shutil.copy2(eda_plot,SUBMISSION_DIR/'curves'/'EDA_class_distribution.png')
shutil.copy2(eda_montage,SUBMISSION_DIR/'curves'/'EDA_sample_images.png')
shutil.copy2(confusion_plot,SUBMISSION_DIR/'curves'/'confusion_F01.png')

# Generate a result-driven report with actual workbook tables and error counts.
def markdown_table(frame, columns=None, limit=None):
    view = frame if columns is None else frame[columns]
    if limit is not None:
        view = view.head(limit)
    headers = list(view.columns)
    lines = ['| ' + ' | '.join(map(str, headers)) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    for row in view.itertuples(index=False, name=None):
        vals = [str(v).replace('|', '\\|').replace('\n', ' ') if pd.notna(v) else '' for v in row]
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines)

final_f1=[m['macro_f1'] for m in final_ms]; baseline_f1=[m['macro_f1'] for m in base_ms]
confusion_pairs=[]
cm_off=cm.copy(); np.fill_diagonal(cm_off,0)
for idx in np.argsort(cm_off.ravel())[::-1][:5]:
    actual, predicted=np.unravel_index(idx,cm_off.shape)
    if cm_off[actual,predicted] > 0:
        confusion_pairs.append(f'{CLASS_NAMES[actual]} ? {CLASS_NAMES[predicted]}: {int(cm_off[actual,predicted])}')
confusion_text='; '.join(confusion_pairs) if confusion_pairs else 'Kh?ng c? l?i ngo?i ???ng ch?o.'
report=f'''# Lab Day 2 ? DeepWeeds

## 1. T?m t?t

- B?i to?n: ph?n lo?i 9 l?p DeepWeeds tr?n fold 0; test ch? ???c d?ng ? giai ?o?n final, m?i seed m?t l??t.
- C?u h?nh ???c ch?n theo validation: `{FINAL_EXP_ID}`; backbone `{ABLATION_BACKBONE}`; recipe `{selected_train_id}` ({selected_changes}); inference `{FINAL_INFERENCE_METHOD}`; temperature scaling `{'on' if FINAL_TEMPERATURE else 'off'}`.
- Test macro-F1 c?a final (mean ? sample std, 3 seed): **{np.mean(final_f1):.4f} ? {np.std(final_f1,ddof=1):.4f}**.
- Baseline T00 test macro-F1: **{np.mean(baseline_f1):.4f} ? {np.std(baseline_f1,ddof=1):.4f}**.
- Ch?nh l?ch final ? baseline: **{np.mean(final_f1)-np.mean(baseline_f1):+.4f}**; v?i 3 seed/1 fold, di?n gi?i th?n tr?ng v? kh?ng xem ch?nh l?ch nh? h?n ?? nhi?u l? c?i thi?n.

## 2. D? li?u v? thi?t l?p

DeepWeeds g?m 17.509 ?nh RGB, 9 nh?n; d?ng split CSV nguy?n b?n fold 0. `images.zip` MD5: `{actual_md5}`. S? ?nh: train {split_stats['n']['train']}, val {split_stats['n']['val']}, test {split_stats['n']['test']}; t?ng unique {split_stats['total_unique']}; overlaps {split_stats['overlap']}.

| M?i tr??ng | Gi? tr? |
|---|---|
| Runtime | Google Colab GPU |
| GPU | {torch.cuda.get_device_name(0)} |
| Python / PyTorch / timm | {platform.python_version()} / {torch.__version__} / {timm.__version__} |
| Input / batch / AMP | 224 px / 32 / FP16 autocast |
| Seeds | screening 0; final 0, 1, 2 |
| Checkpoint | epoch c? macro-F1 validation cao nh?t |

T?p train d?ng ?? c?p nh?t tr?ng s?; validation d?ng ch?n backbone, recipe v? inference; test kh?ng ???c d?ng ?? ch?n. ?? l?ch chu?n qua seed l? sample std (`ddof=1`). EDA class distribution v? ?nh v? d? ???c l?u trong `curves/`.

## 3. So s?nh backbone

C?ng recipe n?n v? seed 0; GMAC ???c ?o b?ng ptflops n?u ki?n tr?c ???c h? tr? (gi? tr? tr?ng/NaN ngh?a l? c?ng c? kh?ng h? tr?, kh?ng ph?i 0).

{markdown_table(frames['Backbones'], ['exp_id','backbone','pretrained_tag','params_M','GMAC','macro_f1_val','top1_val','train_seconds_per_epoch','latency_batch1_ms','notes'])}

Backbone d?ng cho ablation/final ???c ch?n b?ng macro-F1 validation trong v?ng s?ng l?c. Latency l? batch 1 ?o th?t tr?n GPU Colab; kh?ng suy r?ng sang GPU kh?c.

## 4. C?ng th?c hu?n luy?n

T01?T06 m?i l?n thay m?t tr?c so v?i T00; T07 l? ph?p th? k?t h?p augmentation, focal loss v? CutMix. C?c s? s?ng l?c ch? c? m?t seed.

{markdown_table(frames['Training'], ['exp_id','backbone','changed_axis','difference_from_T00','macro_f1_val','top1_val','delta_macro_f1_vs_T00','rare_class_f1_min','notes'])}

K?t qu? ch?a ?? ?? kh?ng ??nh y?u t? n?o th?ng ?n ??nh n?u kh?c bi?t nh?; final mean/std l? b?ng ch?ng b? sung nh?ng v?n ch? t? m?t fold.

## 5. Suy lu?n, hi?u chu?n v? latency

Temperature T ???c fit ch? tr?n validation. Latency c? warmup 10, 50 l?n ?o, CUDA synchronization tr??c/sau, batch 1, FP32; kh?ng t?nh preprocessing.

{markdown_table(frames['Inference'], ['exp_id','method','model_checkpoint','K_views_or_models','macro_f1_val','top1_val','ECE_val','latency_p50_ms','latency_p95_ms','latency_p99_ms','throughput_images_s','relative_cost_vs_I00'])}

Ch?n inference d?a tr?n validation macro-F1, sau ?? ECE v? p95. B?o c?o trade-off b?ng c?c s? p95 v? macro-F1 ? b?ng, kh?ng coi TTA l? mi?n ph?.

## 6. K?t qu? final v? ph?n t?ch l?i

{markdown_table(frames['Final'], ['exp_id','seed','macro_f1_val','macro_f1_test','top1_test','ece_test','row_type'])}

### Ch? s? theo l?p (trung b?nh seed)

{markdown_table(frames['PerClass'], ['exp_id','class','test_support_mean','precision_mean','recall_mean','f1_mean'])}

### C?c l?i nh?m l?n nhi?u nh?t

D?a tr?n confusion matrix t? trung b?nh x?c su?t c?a ba seed: {confusion_text}. H?y xem ?nh t??ng ?ng trong t?p test v? b? sung gi? thuy?t tr?c quan (??c bi?t Chinee Apple ? Snake Weed); kh?ng s? d?ng k?t qu? n?y ?? thay ??i m? h?nh.

## 7. K?t lu?n, khuy?n ngh? v? h?n ch?

Final c? macro-F1 test {np.mean(final_f1):.4f} ? {np.std(final_f1,ddof=1):.4f}, baseline {np.mean(baseline_f1):.4f} ? {np.std(baseline_f1,ddof=1):.4f}. ??ng g?p backbone/recipe/inference c?n ???c di?n gi?i t? c?c b?ng validation ? tr?n; n?u ? nh? so v?i bi?n thi?n gi?a seed th? k?t lu?n l? ch?a ph?n bi?t ???c. V?i tri?n khai robot, ch?n ph??ng ph?p d?a tr?n p95 ?o ???c v? ng?n s?ch th?c t?, kh?ng ngo?i suy latency sang ph?n c?ng kh?c.

Gi?i h?n: ch? m?t fold v? ba seed cu?i; fold chia ng?u nhi?n kh?ng theo ??a ?i?m n?n test c? th? l?c quan so v?i m?a/??a ?i?m m?i. Ng?n s?ch Colab h?n ch? s? seed v? s? bi?n th?. K?t qu? kh?ng b?o ??m chuy?n mi?n.

## Ph? l?c: t?i l?p

- M? ngu?n ch?y ???c v? `eval.py` g?c n?m trong `code/`.
- To?n b? prediction files cu?i/m?c n?m trong `predictions/`; exp_id kh?p v?i workbook v? curves.
- Notebook ch?y qua Google Drive; thay placeholder trong README b?ng link Colab chia s?.
- C?u h?nh ??y ?? t?ng run n?m trong `runs/<exp_id>/seed<k>/config.json`; checkpoint kh?ng ???c ??a v?o submission.
'''
(SUBMISSION_DIR/'report.md').write_text(report,encoding='utf-8')
readme=f'''# DeepWeeds Lab Day 2 ? {STUDENT_ID} {STUDENT_NAME}

- Notebook Colab ch?y l?i: (d?n link Colab t?i ??y)
- Dataset: DeepWeeds fold 0; CSV g?c; train/val/test counts: {split_stats['n']}.
- M?i tr??ng l?n ch?y: Python {platform.python_version()}, torch {torch.__version__}, timm {timm.__version__}, GPU {torch.cuda.get_device_name(0)}.
- Seeds screening: 0; final: 0, 1, 2.
- Ch?y notebook theo th? t? t? ??u ??n cu?i. `eval.py` gi? nguy?n b?n g?c.
- File d? ?o?n test c?a T00/F01 d?ng ??ng ??nh d?ng `Filename,y_true,y_pred,p0..p8`.
- Kh?ng ch?a dataset ho?c checkpoint l?n.
'''
(SUBMISSION_DIR/'README.md').write_text(readme,encoding='utf-8')
print('Submission exported:',SUBMISSION_DIR)
print('Workbook:',xlsx_path)
print('Final test macro-F1:',np.mean(final_f1),'?',np.std(final_f1,ddof=1))
